In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings

warnings.filterwarnings("ignore")

In [3]:
sms_data=pd.read_csv("../Data/spam.csv", encoding="latin-1")

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [ ]:
import pandas as pd
import numpy as np
from typing import Dict, List, Any

class DataValidation:
    """
    A class for validating machine learning datasets, particularly for spam detection tasks.
    Performs checks for data quality, structure, and statistical properties.
    """

    def __init__(self, data: pd.DataFrame):
        """
        Initialize with a pandas DataFrame.

        Args:
            data (pd.DataFrame): The dataset to validate
        """
        self.data = data.copy()
        self.validation_results = {}

    def check_missing_values(self) -> Dict[str, Any]:
        """
        Check for missing values in each column.

        Returns:
            Dict containing missing value statistics
        """
        missing_info = {}
        total_rows = len(self.data)

        for col in self.data.columns:
            missing_count = self.data[col].isnull().sum()
            missing_percentage = (missing_count / total_rows) * 100

            missing_info[col] = {
                'missing_count': missing_count,
                'missing_percentage': missing_percentage,
                'has_missing': missing_count > 0
            }

        self.validation_results['missing_values'] = missing_info
        return missing_info

    def check_data_types(self) -> Dict[str, str]:
        """
        Check data types of each column.

        Returns:
            Dict mapping column names to their data types
        """
        data_types = self.data.dtypes.to_dict()
        self.validation_results['data_types'] = data_types
        return data_types

    def check_unique_values(self, max_unique_threshold: int = 50) -> Dict[str, Any]:
        """
        Check unique values in each column, especially categorical ones.

        Args:
            max_unique_threshold: Maximum number of unique values to list

        Returns:
            Dict with unique value information per column
        """
        unique_info = {}

        for col in self.data.columns:
            unique_vals = self.data[col].unique()
            n_unique = len(unique_vals)

            unique_info[col] = {
                'n_unique': n_unique,
                'unique_values': list(unique_vals) if n_unique <= max_unique_threshold else f'More than {max_unique_threshold} unique values'
            }

        self.validation_results['unique_values'] = unique_info
        return unique_info

    def check_class_balance(self, target_column: str) -> Dict[str, Any]:
        """
        Check class distribution for classification tasks.

        Args:
            target_column: Name of the target/label column

        Returns:
            Dict with class balance information
        """
        if target_column not in self.data.columns:
            raise ValueError(f"Target column '{target_column}' not found in data")

        class_counts = self.data[target_column].value_counts()
        class_percentages = (class_counts / len(self.data)) * 100

        balance_info = {
            'class_counts': class_counts.to_dict(),
            'class_percentages': class_percentages.to_dict(),
            'is_balanced': min(class_percentages) / max(class_percentages) > 0.5  # Simple balance check
        }

        self.validation_results['class_balance'] = balance_info
        return balance_info

    def check_outliers(self, numeric_columns: List[str] = None, method: str = 'iqr') -> Dict[str, Any]:
        """
        Check for outliers in numeric columns using IQR or Z-score method.

        Args:
            numeric_columns: List of numeric columns to check (if None, auto-detect)
            method: 'iqr' or 'zscore'

        Returns:
            Dict with outlier information per column
        """
        if numeric_columns is None:
            numeric_columns = self.data.select_dtypes(include=[np.number]).columns.tolist()

        outlier_info = {}

        for col in numeric_columns:
            if col not in self.data.columns:
                continue

            values = self.data[col].dropna()

            if method == 'iqr':
                Q1 = values.quantile(0.25)
                Q3 = values.quantile(0.75)
                IQR = Q3 - Q1
                lower_bound = Q1 - 1.5 * IQR
                upper_bound = Q3 + 1.5 * IQR

                outliers = ((values < lower_bound) | (values > upper_bound)).sum()
            elif method == 'zscore':
                z_scores = np.abs((values - values.mean()) / values.std())
                outliers = (z_scores > 3).sum()
            else:
                raise ValueError("Method must be 'iqr' or 'zscore'")

            outlier_percentage = (outliers / len(values)) * 100

            outlier_info[col] = {
                'outlier_count': outliers,
                'outlier_percentage': outlier_percentage,
                'method': method
            }

        self.validation_results['outliers'] = outlier_info
        return outlier_info

    def generate_summary_report(self) -> Dict[str, Any]:
        """
        Generate a comprehensive validation summary.

        Returns:
            Dict containing all validation results
        """
        # Run all checks if not already done
        if 'missing_values' not in self.validation_results:
            self.check_missing_values()
        if 'data_types' not in self.validation_results:
            self.check_data_types()
        if 'unique_values' not in self.validation_results:
            self.check_unique_values()

        # Add basic stats
        self.validation_results['basic_stats'] = {
            'total_rows': len(self.data),
            'total_columns': len(self.data.columns),
            'memory_usage': self.data.memory_usage(deep=True).sum()
        }

        return self.validation_results

    def get_validation_issues(self) -> List[str]:
        """
        Identify potential data quality issues.

        Returns:
            List of issue descriptions
        """
        issues = []

        # Check for high missing values
        missing_vals = self.validation_results.get('missing_values', {})
        for col, info in missing_vals.items():
            if info['missing_percentage'] > 20:
                issues.append(f"High missing values in '{col}': {info['missing_percentage']:.1f}%")

        # Check for imbalanced classes (if class balance was checked)
        balance = self.validation_results.get('class_balance', {})
        if not balance.get('is_balanced', True):
            issues.append("Dataset appears imbalanced - consider resampling techniques")

        # Check for high outlier percentages
        outliers = self.validation_results.get('outliers', {})
        for col, info in outliers.items():
            if info['outlier_percentage'] > 10:
                issues.append(f"High outliers in '{col}': {info['outlier_percentage']:.1f}%")

        return issues

In [ ]:
# Step 1: Create DataValidation instance
validator = DataValidation(sms_data)

In [ ]:
# Step 2: Check for missing values
missing_report = validator.check_missing_values()
print("Missing Values Report:")
for col, info in missing_report.items():
    print(f"{col}: {info['missing_count']} missing ({info['missing_percentage']:.2f}%)")

In [ ]:
# Step 3: Check data types
data_types = validator.check_data_types()
print("Data Types:")
for col, dtype in data_types.items():
    print(f"{col}: {dtype}")